# Supervised Fine-tuning (SFT) with MLX

You can run this notebook on any M-series Mac with a minimum of 24GB of RAM.

### What's in this notebook?

In this notebook you will learn how to perform Supervised Fine-tuning (SFT) using MLX for efficient, memory-optimized training on apple silicon.
We will use the [LFM2.5-1.2B-Instruct](https://docs.liquid.ai/docs/models/lfm25-1.2b-instruct) model and fine-tune it on the Smol-SmolTalk dataset using LoRA adapters and MLX's optimizations for faster training with reduced memory consumption.

We will cover
- Environment setup
- Data preparation
- Model training with MLX-LM-LoRA's SFTTrainer
- LoRA for parameter-efficient fine-tuning
- Local inference with your new model
- Model saving and exporting it into the format you need for **deployment**.

### Deployment options

LFM2.5 models are small and efficient, enabling deployment across a wide range of platforms:

<table align="left">
  <tr>
    <th>Deployment Target</th>
    <th>Use Case</th>
  </tr>
  <tr>
    <td>📱 <a href="https://docs.liquid.ai/leap/edge-sdk/android/android-quick-start-guide"><b>Android</b></a></td>
    <td>Mobile apps on Android devices</td>
  </tr>
  <tr>
    <td>📱 <a href="https://docs.liquid.ai/leap/edge-sdk/ios/ios-quick-start-guide"><b>iOS</b></a></td>
    <td>Mobile apps on iPhone/iPad</td>
  </tr>
  <tr>
    <td>🍎 <a href="https://docs.liquid.ai/docs/inference/mlx"><b>Apple Silicon Mac</b></a></td>
    <td>Local inference on Mac with MLX</td>
  </tr>
  <tr>
    <td>🦙 <a href="https://docs.liquid.ai/docs/inference/llama-cpp"><b>llama.cpp</b></a></td>
    <td>Local deployments on any hardware</td>
  </tr>
  <tr>
    <td>🦙 <a href="https://docs.liquid.ai/docs/inference/ollama"><b>Ollama</b></a></td>
    <td>Local inference with easy setup</td>
  </tr>
  <tr>
    <td>🖥️ <a href="https://docs.liquid.ai/docs/inference/lm-studio"><b>LM Studio</b></a></td>
    <td>Desktop app for local inference</td>
  </tr>
  <tr>
    <td>⚡ <a href="https://docs.liquid.ai/docs/inference/vllm"><b>vLLM</b></a></td>
    <td>Cloud deployments with high throughput</td>
  </tr>
  <tr>
    <td>☁️ <a href="https://docs.liquid.ai/docs/inference/modal-deployment"><b>Modal</b></a></td>
    <td>Serverless cloud deployment</td>
  </tr>
  <tr>
    <td>🏗️ <a href="https://docs.liquid.ai/docs/inference/baseten-deployment"><b>Baseten</b></a></td>
    <td>Production ML infrastructure</td>
  </tr>
  <tr>
    <td>🚀 <a href="https://docs.liquid.ai/docs/inference/fal-deployment"><b>Fal</b></a></td>
    <td>Fast inference API</td>
  </tr>
</table>

### Need help building with our models and tools?
Join the Liquid AI Discord Community and ask.

<a href="https://discord.com/invite/liquid-ai"><img src="https://img.shields.io/discord/1385439864920739850?color=7289da&label=Join%20Discord&logo=discord&logoColor=white" alt="Join Discord"></a>

And now, let the fine tune begin!

# 📦 Installation & Setup

First, let's install all the required packages.

In [ ]:
%%capture
!pip install mlx-lm-lora

Let's now verify the packages are installed correctly

In [ ]:
from mlx_lm_lora.utils import from_pretrained, save_pretrained_merged, calculate_iters
from mlx_lm_lora.trainer.sft_trainer import SFTTrainingArgs, train_sft
from mlx_lm_lora.trainer.datasets import CacheDataset, TextDataset
from datasets import load_dataset

from mlx_lm.tuner.utils import print_trainable_parameters, build_schedule

import mlx.optimizers as optim

# Loading the model from MLX-LM-LoRA



In [ ]:
# Select a model to fine-tune from the list
lfm_models = [
    "LiquidAI/LFM2.5-1.2B-Instruct",
    "LiquidAI/LFM2.5-1.2B-JP",
    "LiquidAI/LFM2-8B-A1B",
    "LiquidAI/LFM2-2.6B-Exp",
    "LiquidAI/LFM2-2.6B",
    "LiquidAI/LFM2-700M",
    "LiquidAI/LFM2-350M",
]

# Model to fine-tune
model_id = "LiquidAI/LFM2.5-1.2B-Instruct"
new_model_name = "lfm2-sft"

# LoRA adapter configuration
lora_config = {
    "rank": 12,  # Low-rank bottleneck size (Larger rank = smarter, but slower). Suggested 8, 16, 32, 64, 128
    "dropout": 0.0,
    "scale": 10.0, # Multiplier for how hard the LoRA update hits the base weights
    "use_dora": False,
    "num_layers": 12 # Use -1 for all layers
}

# Quantization configuration
quantized_config = {
    "bits": 4,
    "group_size": 64,
    "mode": "affine",
}

# Load the model and tokenizer
model, tokenizer, adapter_file = from_pretrained(
    model=model_id,
    lora_config=lora_config,
    quantized_load=quantized_config,
    new_adapter_path=f"./{new_model_name}"
)
print_trainable_parameters(model)

# 🎯 Part 1: Supervised Fine-Tuning (SFT)

SFT teaches the model to follow instructions by training on input-output pairs (instruction vs response). This is the foundation for creating instruction-following models.

## Load an SFT Dataset

We will use [HuggingFaceTB/smol-smoltalk](https://huggingface.co/datasets/HuggingFaceTB/smol-smoltalk), limiting ourselves to the first 1k samples for brevity. Feel free to change the limit by changing the slicing index in the parameter `split`.

In [ ]:
def format(sample):
    sample["text"] = tokenizer.apply_chat_template(
        sample["messages"],
        add_generation_prompt=False,
        tokenize=False,
    )
    return sample

print("📥 Loading SFT dataset...")
train_dataset_sft = load_dataset("HuggingFaceTB/smol-smoltalk", split="train[:1000]").map(format)
eval_dataset_sft = load_dataset("HuggingFaceTB/smol-smoltalk", split="test[:50]").map(format)

train_set = TextDataset(train_dataset_sft, tokenizer, text_key="text")
eval_set = TextDataset(eval_dataset_sft, tokenizer, text_key="text")

print("✅ SFT Dataset loaded:")
print(f"   📚 Train samples: {len(train_set)}")
print(f"   🧪 Eval samples: {len(eval_set)}")
print(f"\n📝 Single Sample: {train_dataset_sft[0]['text']}")

## Launch Training: Quantization + LoRA + SFT
We are now ready to launch an SFT run wraped in LoRA (Low-Rank Adaptation) which allows efficient fine-tuning by only training a small number of additional parameters. Perfect for limited compute resources, feel free to modify `SFTTrainingArgs` to play around with different configurations.

In [ ]:
batch_size = 1 # Number of training samples processed in each forward/backward pass. A batch size of 1 minimizes memory usage.
epochs = 1 # Number of complete passes through the training dataset.

# Convert the desired number of epochs into the number of training iterations.
iters = calculate_iters(
    train_set,
    batch_size=batch_size,
    epochs=epochs
)

lr = build_schedule(
    schedule_config={
        "name": "cosine_decay", # Cosine decay gradually reduces the learning rate during training.
        "warmup": 40, # Number of initial steps used to gradually increase the learning rate. Warmup helps avoid unstable updates at the beginning of training.
        "warmup_init": 2e-7, # Learning rate used at the very beginning of the warmup period.
        "arguments": [
            2e-5,               # Peak learning rate after warmup
            int(iters * 0.95),  # Decay LR over roughly 95% of training
            2e-6                # Final/minimum learning rate
        ],
    }
)

opt = optim.AdamW(
    learning_rate=lr, # Uses the dynamic learning-rate schedule defined above.
    betas=[0.9, 0.999], # Adam momentum coefficients. beta1 controls the moving average of gradients. beta2 controls the moving average of squared gradients.
    eps=1e-8, # Small numerical-stability constant used by AdamW.
    weight_decay=0.00, # Strength of weight decay regularization. 0.00 disables weight decay.
    bias_correction=False # Disables Adam's bias correction for the moving averages.
)

print("🏗️  Creating SFT trainer configuration...")
sft_config = SFTTrainingArgs(
    # Training objective.
    # "nll" = Negative Log-Likelihood: maximize the probability of the
    # correct target tokens provided by the supervised training data.
    loss_type="nll",
    batch_size=batch_size, 
    iters=iters,
    gradient_accumulation_steps=4, # Accumulate gradients across 4 micro-batches before updating weights. With batch_size=1, this gives an effective batch size of 4 samples.
    val_batches=1,  # Number of validation batches used during each evaluation. Keeping this small makes evaluation faster but also noisier.
    steps_per_report=100, # Print/log training metrics every 200 steps.
    steps_per_eval=200, # Run validation every 400 steps.
    steps_per_save=200, # Save a training checkpoint every 400 steps.
    max_seq_length=1024, # Maximum sequence length used during training. Sequences longer than 1024 tokens are truncated/handled according to the trainer's sequence-processing behavior.
    adapter_file=adapter_file, # File/path where the trained adapter weights are stored.
    grad_checkpoint=True, # Enable gradient checkpointing. Saves memory by recomputing some activations during backward instead of storing all of them during the forward pass. Trade-off: lower memory usage, slightly more computation.
    seq_step_size=None, # Optional sequence chunk/step size. None means no custom sequence stepping/chunking is configured.

    # Quantization-Aware Training (QAT)
    # Disable Quantization-Aware Training for this run.
    # The QAT settings below therefore have no effect unless this is True.
    qat_enable=False,
    qat_bits=quantized_config["bits"], # Target number of bits used by QAT.
    qat_group_size=quantized_config["bits"], # Number of parameters grouped together for quantization.
    qat_mode=quantized_config["mode"], # Quantization mode/strategy used during QAT.
    qat_start_step=1, # Training step at which QAT should begin.
    qat_interval=1, # Apply/update QAT behavior every N training steps.
)

print("\n🚀 Starting SFT training...")
train_sft(
    model=model, # Model whose parameters/adapters will be trained.
    args=sft_config, # SFT configuration defined above.
    optimizer=opt, # AdamW optimizer and LR schedule.
    train_dataset=CacheDataset(train_set), # Cached training dataset to reduce repeated preprocessing work.
    val_dataset=CacheDataset(eval_set), # Cached validation dataset used during evaluation.
)
print("🎉 SFT training completed!")

## Save merged model

Merge the extra weights learned with LoRA back into the model to obtain a "normal" model checkpoint.

In [ ]:
print("\n🔄 Merging and save LoRA weights...")
save_pretrained_merged(
    model=model, # Trained model.
    tokenizer=tokenizer, # Tokenizer saved alongside the model.
    save_path=new_model_name, # Directory/name for the final merged model.
    de_quantize=True, # Convert quantized weights back to regular weights when saving. You have to turn it to false when qat is enabled.
    remove_adapters=True # Merge/remove adapter structure so the result is a standalone model rather than a base model that requires separate adapter files.
)
print(f"💾 SFT Merged model saved to: {sft_config.adapter_file}")